# Imports

* 무거운 "train" 작업 전체의 input의 output만 보고하는 튜토리얼 작성하고
* 그러고 나면 다음 단계로, train과정에서 들어가는 함수들을 쪼개서 분석해보는 tutorial도 해보기
  * `train.py`에서 호출하는 세부 모듈을 그대로 가져다 쓰되, train.py의 전체 흐름을 조금 더 가볍게 뫄하는 별도 스크립트 작성이 필요할 것으로 보임.


  train.py의 input이 뭔지 파악이 되고 나면 전체 프로그램의 흐름(모듈 간 의존관계)를 다이어그램(ppt)으로 그려보기
  * 핵심은, "특정 기능만 사용/분석 하려고 하면 뭐는 몰라도 되느냐"

  생각날 때 틈틈이:
     * 3DGS 발전 방향 고민 및 최근 연구 탐색(검색)해보기
     * multi-view 이미지를 3DGS 형태로 복원 --> mesh 형태로 변환하는

In [2]:
import os
import torch
import numpy as np
import open3d as o3d
from random import randint
from utils.loss_utils import l1_loss, ssim
from gaussian_renderer import render, network_gui
import sys
from scene import Scene, GaussianModel
from utils.general_utils import safe_state, get_expon_lr_func
import uuid
from tqdm import tqdm
from utils.image_utils import psnr
from argparse import ArgumentParser, Namespace
from arguments import ModelParams, PipelineParams, OptimizationParams
from scene.dataset_readers import sceneLoadTypeCallbacks
try:
    from torch.utils.tensorboard import SummaryWriter
    TENSORBOARD_FOUND = True
except ImportError:
    TENSORBOARD_FOUND = False

try:
    from fused_ssim import fused_ssim
    FUSED_SSIM_AVAILABLE = True
except:
    FUSED_SSIM_AVAILABLE = False

try:
    from diff_gaussian_rasterization import SparseGaussianAdam
    SPARSE_ADAM_AVAILABLE = True
except:
    SPARSE_ADAM_AVAILABLE = False

Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


# Running train.py

We will run the following terminal command:

```cmd
python train.py -s Gaussiatest/Test2
```

Here this command assumes there are already point3D.ply which is the output of convet.py in [GaussianTest/Test2/sparse/0](GaussianTest/Test2/sparse/0)

# Input

`train.py`에 정의되어 있는 `training`이라는 함수에  

gaussians = GaussianModel(dataset.sh_degree, opt.optimizer_type)  
scene = Scene(dataset, gaussians)

위와 같은 코드가 있는데 `Scene` 클래스에서 `scene.dataset_readers`의 `sceneLoadTypeCallbacks["Colmap"]`라는 메소드로 [tutorial_convert_py.ipynb](tutorial_convert_py.ipynb)에서 했던 것과 같이 `Image Undistorter`가 진행된 후의 images.bin 파일과 cameras.bin 파일을 읽어온다. 이 값들을 `scene_info` 변수가 가져오고 `prepare_output_and_logger` 메소드가 설정한 [args.model_path](output/Test2_Result)에 `input.ply`(새로 만들어짐) 파일로 [GaussianTest/Test2/sparse/0](GaussianTest/Test2/sparse/0)/point3D.ply의 값을 복사하여 가져온다. 이를 GaussianModel 클래스의 `create_from_pcd` 메소드를 이용해 3D Gaussian을 만든다.

## Input.ply

In [3]:
ply_path = 'output/Test2_Result/input.ply'
pcd = o3d.io.read_point_cloud(ply_path)

print(pcd)
points = np.asarray(pcd.points)
print(points)

o3d.visualization.draw_geometries([pcd])

PointCloud with 1768 points.
[[  4.34101534  -1.97590983   8.94548512]
 [  6.54583645  10.24704552   2.88442421]
 [  4.26116467  -0.91624063   9.97420979]
 ...
 [-14.37703228  10.57718468   7.03714371]
 [ -3.41339111   4.91355276  12.40224266]
 [-14.48631001  10.56289577   6.95793295]]


# Outputs

By running train.py its ouputs are stored in directory [output/Test2_Result](output/Test2_Result)

### Details

Information [input.ply](output/Test2_Result/input.ply) have is the result of running convert.py. Which is 1768 point clouds.

## Results for 7000 iterations

In [4]:
ply_path = 'output/Test2_Result/point_cloud/iteration_7000/point_cloud.ply'
pcd = o3d.io.read_point_cloud(ply_path)

print(pcd)
points = np.asarray(pcd.points)
print(points)

o3d.visualization.draw_geometries([pcd])

PointCloud with 347527 points.
[[ 6.59872437 10.12901211  2.86574125]
 [ 0.22425154  1.65513897 12.2174387 ]
 [ 4.22987652 -1.58260691  9.34566116]
 ...
 [ 0.07556491  6.75840139  6.57672215]
 [ 4.2516675   7.14003801 10.84717178]
 [-0.48998436  5.14745998  7.08731127]]


## Results for 30000 iterations

In [5]:
ply_path = 'output/Test2_Result/point_cloud/iteration_30000/point_cloud.ply'
pcd = o3d.io.read_point_cloud(ply_path)

print(pcd)
points = np.asarray(pcd.points)
print(points)

o3d.visualization.draw_geometries([pcd])

PointCloud with 375184 points.
[[ 6.60873985 10.11532307  2.88246822]
 [ 0.17242943  1.56833577 12.10415363]
 [ 4.23568726 -1.58306515  9.3474226 ]
 ...
 [-1.06505299  5.10284138 12.32131481]
 [ 1.89028597  3.38411283 10.96808529]
 [-0.26036382  4.30767822 11.26516533]]
